# LLMAssessment2 cloud smoke test

Runs the actual setup/function cells from `with_rag_v6.ipynb`, then generates one short answer (80 tokens maximum). Upload this notebook, `with_rag_v6.ipynb`, and `papers.json` to the Colab runtime before running.

In [ ]:
!pip install -q transformers torch accelerate sentence-transformers faiss-cpu
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import json, os, time

assert os.path.exists('/content/with_rag_v6.ipynb'), 'Upload with_rag_v6.ipynb first'
assert os.path.exists('/content/papers.json'), 'Upload papers.json first'
with open('/content/with_rag_v6.ipynb', encoding='utf-8') as f:
    final_nb = json.load(f)

# Execute the final notebook's real code cells, skipping pip and the 10-question batch.
for idx in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]:
    src = ''.join(final_nb['cells'][idx]['source'])
    print(f'\n--- executing final notebook Cell {idx} ---')
    exec(compile(src, f'with_rag_v6.ipynb:Cell {idx}', 'exec'), globals())
print('\nFINAL NOTEBOOK SETUP: PASS')

In [ ]:
MAX_NEW_TOKENS = 80
case = TEST_CASES[0]
t0 = time.time()
reply = chat(case['question'])
elapsed = time.time() - t0
record = {
    'answer': reply['answer'],
    'case': case,
    'retrieved_titles': reply['retrieved_titles'],
    'latency': elapsed,
}
metrics = evaluate(record)
assert reply['answer'].strip(), 'Empty model answer'
assert len(reply['retrieved_titles']) > 0, 'No retrieved papers'
print('CLOUD_SMOKE_PASS')
print('retrieved_titles:', reply['retrieved_titles'])
print('retrieval_hit:', metrics['retrieval_hit'])
print('keyword_recall:', round(metrics['keyword_recall'], 3))
print('latency_seconds:', round(elapsed, 2))
print('answer:', reply['answer'])